<a href="https://colab.research.google.com/github/marinhomar/cardioia-fase2/blob/main/src/parte2/parte2_classificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CardioIA – Parte 2: Classificador de risco (alto x baixo)

## 1. Bibliotecas

In [ ]:
# pandas: para trabalhar com tabelas
import pandas as pd

# separa os dados em uma parte para treinar e outra para testar
from sklearn.model_selection import train_test_split

# TF-IDF: transforma as frases em números que o computador entende
from sklearn.feature_extraction.text import TfidfVectorizer

# o modelo que vai aprender a classificar o risco
from sklearn.linear_model import LogisticRegression

# para medir quantas o modelo acertou
from sklearn.metrics import accuracy_score

## 2. Leitura da base de frases

In [ ]:
# endereço do arquivo no GitHub
URL = "https://raw.githubusercontent.com/marinhomar/cardioia-fase2/main/src/parte2/frases_risco.csv"

# lê as 40 frases com o rótulo de risco
dados = pd.read_csv(URL)

# confere se a base está equilibrada (20 alto e 20 baixo)
dados["situacao"].value_counts()

,count
situacao,
alto risco,20
baixo risco,20


## 3. Separação entre treino e teste

In [ ]:
# 75% das frases para o modelo aprender (30) e 25% para a prova (10)
# stratify: mantém a mesma proporção de alto e baixo risco nas duas partes
# random_state: garante que a divisão seja sempre a mesma toda vez que rodar
X_treino, X_teste, y_treino, y_teste = train_test_split(
    dados["frase"], dados["situacao"],
    test_size=0.25, random_state=42, stratify=dados["situacao"]
)

print("Frases para treino:", len(X_treino))
print("Frases para teste:", len(X_teste))

Frases para treino: 30
Frases para teste: 10


## 4. TF-IDF: transformando frases em números

In [ ]:
# cria o tradutor de texto para números
tfidf = TfidfVectorizer()

# aprende o vocabulário só com as frases de treino e já transforma
X_treino_vet = tfidf.fit_transform(X_treino)

# transforma as frases de teste usando o mesmo vocabulário (sem aprender nada novo)
X_teste_vet = tfidf.transform(X_teste)

print("Tamanho da tabela de treino (frases x palavras):", X_treino_vet.shape)

Tamanho da tabela de treino (frases x palavras): (30, 123)


## 5. Treinamento do modelo

In [ ]:
# cria o modelo de Regressão Logística
modelo = LogisticRegression()

# o modelo aprende quais palavras puxam para alto ou baixo risco
modelo.fit(X_treino_vet, y_treino)

LogisticRegression()

## 6. Avaliação: acurácia

In [ ]:
# o modelo tenta classificar as 10 frases que nunca viu
previsoes = modelo.predict(X_teste_vet)

# compara com a resposta certa e calcula a porcentagem de acerto
acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia: {acuracia:.0%}")

Acurácia: 90%


## 7. Conferência frase a frase

In [ ]:
# mostra cada frase de teste com a resposta certa e o que o modelo disse
comparacao = pd.DataFrame({"frase": X_teste, "real": y_teste, "previsto": previsoes})
comparacao["acertou"] = comparacao["real"] == comparacao["previsto"]
comparacao

## 8. Teste ao vivo

In [ ]:
# @title Teste ao vivo: digite uma frase para classificar o risco
frase = "estou com muita dor no peito e falta de ar"  # @param {type:"string"}

# transforma a frase em números com o mesmo TF-IDF do treino
frase_vet = tfidf.transform([frase])

# o modelo classifica e mostra o quanto tem certeza
risco = modelo.predict(frase_vet)[0]
confianca = modelo.predict_proba(frase_vet).max()
print(f"Risco: {risco} (confiança: {confianca:.0%})")